## Install NBA API

In [62]:
!pip install -q nba_api

## Import Libraries

In [63]:
from nba_api.stats.static import players
from nba_api.stats.endpoints import playergamelog
import pandas as pd
import time

## Retrieve Player Game Logs

In [64]:
# Pull game logs for a sample set of players
player_list = players.get_players()[:50]

all_games = []

for player in player_list:
    try:
        gamelog = playergamelog.PlayerGameLog(
            player_id=player["id"],
            season="2024-25"
        )

        df_player = gamelog.get_data_frames()[0]
        df_player["PLAYER_NAME"] = player["full_name"]
        all_games.append(df_player)

        time.sleep(0.6)
    except Exception:
        continue

## Combine Player Data

In [65]:
# Combine all player game logs into one dataframe
df = pd.concat(all_games, ignore_index=True)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 322
Columns: 28


/tmp/ipykernel_175/1134098795.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(all_games, ignore_index=True)


,SEASON_ID,Player_ID,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,...,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,PLAYER_NAME
0,22024,1630173,0022401188,"Apr 13, 2025",NYK @ BKN,W,33,8,18,0.444,...,9,2,2,0,2,4,18,19,1,Precious Achiuwa
1,22024,1630173,0022401175,"Apr 11, 2025",NYK vs. CLE,L,15,0,5,0.000,...,6,0,2,1,1,2,0,-9,1,Precious Achiuwa
2,22024,1630173,0022401167,"Apr 10, 2025",NYK @ DET,L,40,8,13,0.615,...,10,3,2,3,0,3,18,-7,1,Precious Achiuwa
3,22024,1630173,0022401128,"Apr 05, 2025",NYK @ ATL,W,25,3,8,0.375,...,3,0,0,2,1,2,6,-10,1,Precious Achiuwa
4,22024,1630173,0022401103,"Apr 02, 2025",NYK @ CLE,L,21,5,9,0.556,...,6,1,1,2,0,1,13,-10,1,Precious Achiuwa


## Prepare and Sort Dataset

In [66]:
# Prepare and sort the dataset
df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
df = df.sort_values(["PLAYER_NAME", "GAME_DATE"]).reset_index(drop=True)

df.head()

,SEASON_ID,Player_ID,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,...,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,PLAYER_NAME
0,22024,1628389,0022400065,2024-10-23,MIA vs. ORL,L,26,1,5,0.200,...,5,1,0,0,0,1,9,-30,1,Bam Adebayo
1,22024,1628389,0022400088,2024-10-26,MIA @ CHA,W,33,6,17,0.353,...,11,3,1,1,1,1,12,0,1,Bam Adebayo
2,22024,1628389,0022400105,2024-10-28,MIA vs. DET,W,35,5,9,0.556,...,10,4,3,3,1,1,12,-3,1,Bam Adebayo
3,22024,1628389,0022400122,2024-10-30,MIA vs. NYK,L,33,3,7,0.429,...,3,4,1,0,1,3,11,-9,1,Bam Adebayo
4,22024,1628389,0022400147,2024-11-02,MIA @ WAS,W,31,12,24,0.500,...,14,2,1,1,1,3,32,25,1,Bam Adebayo


## Create Game Score

In [67]:
# Calculate Game Score
df["gmsc"] = (
    df["PTS"]
    + 0.4 * df["FGM"]
    - 0.7 * df["FGA"]
    - 0.4 * (df["FTA"] - df["FTM"])
    + 0.7 * df["OREB"]
    + 0.3 * df["DREB"]
    + df["STL"]
    + 0.7 * df["AST"]
    + 0.7 * df["BLK"]
    - 0.4 * df["PF"]
    - df["TOV"]
)

df[["PLAYER_NAME", "GAME_DATE", "PTS", "gmsc"]].head()

,PLAYER_NAME,GAME_DATE,PTS,gmsc
0,Bam Adebayo,2024-10-23,9,7.7
1,Bam Adebayo,2024-10-26,12,9.4
2,Bam Adebayo,2024-10-28,12,16.8
3,Bam Adebayo,2024-10-30,11,9.8
4,Bam Adebayo,2024-11-02,32,26.7


## Generate Rolling Features

In [68]:
# Create rolling features using only prior games
df["last5_pts"] = df.groupby("PLAYER_NAME")["PTS"].transform(lambda x: x.shift(1).rolling(5).mean())
df["last10_pts"] = df.groupby("PLAYER_NAME")["PTS"].transform(lambda x: x.shift(1).rolling(10).mean())

df["last5_gmsc"] = df.groupby("PLAYER_NAME")["gmsc"].transform(lambda x: x.shift(1).rolling(5).mean())
df["last10_gmsc"] = df.groupby("PLAYER_NAME")["gmsc"].transform(lambda x: x.shift(1).rolling(10).mean())

df["last5_minutes"] = df.groupby("PLAYER_NAME")["MIN"].transform(lambda x: x.shift(1).rolling(5).mean())
df["last5_fg_pct"] = df.groupby("PLAYER_NAME")["FG_PCT"].transform(lambda x: x.shift(1).rolling(5).mean())

df[
    [
        "PLAYER_NAME",
        "GAME_DATE",
        "PTS",
        "last5_pts",
        "last10_pts",
        "last5_gmsc",
        "last10_gmsc",
        "last5_minutes",
        "last5_fg_pct"
    ]
].head(12)

,PLAYER_NAME,GAME_DATE,PTS,last5_pts,last10_pts,last5_gmsc,last10_gmsc,last5_minutes,last5_fg_pct
0,Bam Adebayo,2024-10-23,9,NaN,NaN,NaN,NaN,NaN,NaN
1,Bam Adebayo,2024-10-26,12,NaN,NaN,NaN,NaN,NaN,NaN
2,Bam Adebayo,2024-10-28,12,NaN,NaN,NaN,NaN,NaN,NaN
3,Bam Adebayo,2024-10-30,11,NaN,NaN,NaN,NaN,NaN,NaN
4,Bam Adebayo,2024-11-02,32,NaN,NaN,NaN,NaN,NaN,NaN
5,Bam Adebayo,2024-11-04,16,15.2,NaN,14.08,NaN,31.6,0.4076
6,Bam Adebayo,2024-11-06,12,16.6,NaN,16.46,NaN,33.6,0.4610
7,Bam Adebayo,2024-11-08,20,16.6,NaN,17.34,NaN,33.8,0.4380
8,Bam Adebayo,2024-11-10,9,18.2,NaN,16.76,NaN,34.6,0.4046
9,Bam Adebayo,2024-11-12,20,17.8,NaN,16.12,NaN,35.8,0.3734


## Create Modeling Dataset

In [75]:
# Turnovers for usage proxy
df["last5_tov"] = df.groupby("PLAYER_NAME")["TOV"].transform(
    lambda x: x.shift(1).rolling(5).mean()
)

# Shot volume
df["last5_fga"] = df.groupby("PLAYER_NAME")["FGA"].transform(
    lambda x: x.shift(1).rolling(5).mean()
)
df["last10_fga"] = df.groupby("PLAYER_NAME")["FGA"].transform(
    lambda x: x.shift(1).rolling(10).mean()
)

# Free throw attempts
df["last5_fta"] = df.groupby("PLAYER_NAME")["FTA"].transform(
    lambda x: x.shift(1).rolling(5).mean()
)

# Usage proxy
df["last5_usage"] = df["last5_fga"] + df["last5_fta"] + df["last5_tov"]

# Player scoring baseline
df["player_avg_pts"] = df.groupby("PLAYER_NAME")["PTS"].transform(
    lambda x: x.shift(1).expanding().mean()
)

# Longer minutes trend
df["last10_minutes"] = df.groupby("PLAYER_NAME")["MIN"].transform(
    lambda x: x.shift(1).rolling(10).mean()
)

# Remove rows where required features are not yet available
df_features = df.dropna(
subset=[
    "player_avg_pts",
    "last5_pts",
    "last5_gmsc",
    "last5_minutes",
    "last5_fga",
    "last5_fta"
]
).reset_index(drop=True)

# Preview key columns
df_features[
    [
        "PLAYER_NAME",
        "GAME_DATE",
        "PTS",
        "player_avg_pts",
        "last5_pts",
        "last10_pts",
        "last5_fga",
        "last10_fga",
        "last5_fta",
        "last5_usage",
        "last5_minutes",
        "last10_minutes",
        "last5_gmsc",
        "last10_gmsc",
        "last5_fg_pct"
    ]
].head()

,PLAYER_NAME,GAME_DATE,PTS,player_avg_pts,last5_pts,last10_pts,last5_fga,last10_fga,last5_fta,last5_usage,last5_minutes,last10_minutes,last5_gmsc,last10_gmsc,last5_fg_pct
0,Bam Adebayo,2024-11-04,16,15.200000,15.2,NaN,12.4,NaN,5.4,18.6,31.6,NaN,14.08,NaN,0.4076
1,Bam Adebayo,2024-11-06,12,15.333333,16.6,NaN,14.4,NaN,4.4,19.6,33.6,NaN,16.46,NaN,0.4610
2,Bam Adebayo,2024-11-08,20,14.857143,16.6,NaN,15.2,NaN,4.6,20.6,33.8,NaN,17.34,NaN,0.4380
3,Bam Adebayo,2024-11-10,9,15.500000,18.2,NaN,17.0,NaN,5.0,23.2,34.6,NaN,16.76,NaN,0.4046
4,Bam Adebayo,2024-11-12,20,14.777778,17.8,NaN,17.8,NaN,5.0,24.6,35.8,NaN,16.12,NaN,0.3734


## Define Features and Target

In [76]:
# Define feature columns
feature_columns = [
    "player_avg_pts",
    "last5_pts",
    "last5_fga",
    "last5_fta",      # ADD THIS
    "last5_minutes",
    "last5_gmsc"
]
# Create model dataset
model_df = df_features.copy()

X = model_df[feature_columns]
y_points = model_df["PTS"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y_points.shape)

X.head()

Feature matrix shape: (297, 6)
Target shape: (297,)


,player_avg_pts,last5_pts,last5_fga,last5_fta,last5_minutes,last5_gmsc
0,15.200000,15.2,12.4,5.4,31.6,14.08
1,15.333333,16.6,14.4,4.4,33.6,16.46
2,14.857143,16.6,15.2,4.6,33.8,17.34
3,15.500000,18.2,17.0,5.0,34.6,16.76
4,14.777778,17.8,17.8,5.0,35.8,16.12


## Split Training and Testing Data

In [77]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_points,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (237, 6)
X_test shape: (60, 6)
y_train shape: (237,)
y_test shape: (60,)


## Train and Evaluate Points Regression Model

In [78]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Train the regression model
rf_points = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

rf_points.fit(X_train, y_train)

# Generate predictions
y_pred = rf_points.predict(X_test)

# Evaluate model performance
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R^2:", r2_score(y_test, y_pred))

MAE: 4.884026162934674
RMSE: 6.1526122077688905
R^2: 0.25061345208400077


## Save Model and Test Prediction

In [79]:
import joblib
import os

# Create models folder if it does not exist
os.makedirs("models", exist_ok=True)

# Save the trained points regression model
joblib.dump(rf_points, "models/points_regression.pkl")

# Test a single prediction
sample = X_test.iloc[[0]]

predicted_points = rf_points.predict(sample)[0]
actual_points = y_test.iloc[0]

print("Predicted points:", round(predicted_points, 2))
print("Actual points:", actual_points)

Predicted points: 5.46
Actual points: 2


In [80]:
#from google.colab import files
#files.download("models/points_regression.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [81]:
# Estimate prediction uncertainty from test residuals
#residuals = y_test - y_pred
#points_std = residuals.std()

#print("Points prediction std dev:", points_std)

Points prediction std dev: 6.142246762206862


In [82]:
#import json
#import os

#os.makedirs("models", exist_ok=True)

#with open("models/points_model_stats.json", "w") as f:
    json.dump({"std_dev": float(points_std)}, f)